# relay_board

> Phase 23 (`specs/multiagent/phase23/`, experiment C1b): bridges that carry evidence. `RelayBoardEnv` is `BridgeBoardEnv` with three keys (`RELAY_KEYS`, all off by default). `evidence`: once a bridge has relayed any value on a coupled subtask, the receiving squad's board shows and counts that clue's whole mirror tally, live, with mirror senders only (P23-2). `mirrorWindow`: no bridges; the receiving squad sees every coupled clue's mirror tally from the start. `bridgeRole`: the bridges are dedicated (`roleBridges`); the role is a prompt rule, not enforced by the board (P23-3). `relaySenders` and `relayTally` are what `clue_colony.tally` merges; `countedFalse` is what `colony_trace.falseValues` adds, so contamination counts lies that arrive through relays and windows. `checkRelay`, `relayKw` and `boardClass` pick the board. Without a relay key every board is unchanged.

In [ ]:
#| default_exp relay_board

In [ ]:
#| hide
from nbdev.showdoc import *

## Keys and checks

`checkRelay` refuses a window with bridges, a dedicated bridge without bridges or on a guarding bridge, and any relay key without `couple` (FR-A6). `boardClass(world)` is the board every run path builds: `RelayBoardEnv` with a relay key, `BridgeBoardEnv` with a bridge key, else `None` (the caller's own default).

In [ ]:
#| export
from explRL.bridge_board import BRIDGE_KEYS, BridgeBoardEnv

RELAY_KEYS = ('evidence', 'bridgeRole', 'mirrorWindow')


def checkRelay(world, N=None, nLiar=0):
    "FR-A6: mirrorWindow needs bridges = 0; bridgeRole needs bridges >= 1 and not bridgeGuard; every relay key needs couple."
    on = [k for k in RELAY_KEYS if world.get(k)]
    if not on:
        return True
    b = world.get('bridges', 0)
    if not world.get('couple'):
        raise ValueError(f'{on[0]} needs couple')
    if world.get('mirrorWindow') and b:
        raise ValueError('mirrorWindow needs bridges = 0 (the window replaces the bridge)')
    if world.get('bridgeRole') and (not b or world.get('bridgeGuard')):
        raise ValueError('bridgeRole needs bridges >= 1 and not bridgeGuard')
    return True


def relayKw(world):
    "The relay keys set in a world, as RelayBoardEnv's keyword arguments ({} on any other world)."
    return {k: bool(world[k]) for k in RELAY_KEYS if world.get(k)}


def boardClass(world):
    "RelayBoardEnv with a relay key, BridgeBoardEnv with a bridge key, else None."
    return RelayBoardEnv if relayKw(world) else BridgeBoardEnv if any(k in world for k in BRIDGE_KEYS) else None

## The board

`relayTally(k)` is the mirror tally of clue 1 of a coupled subtask, `[(r, senders, newest)]`, exposed to the receiving squad when the clue is open to it: on a `mirrorWindow` board always, on an `evidence` board once `relayOpened(k)` (any relay on k), otherwise `[]`. `relaySenders(k, j)` is the same as `{r: senders}` (`{}` for j ≠ 1). `countedFalse()` gives, for every false mirror value a receiving board has shown, the (time, agent) of the event that first exposed it: its first mirror post, or on an evidence board the first relay on k if that came later.

In [ ]:
#| export
class RelayBoardEnv(BridgeBoardEnv):
    """BridgeBoardEnv with Phase 23's keys (FR-A1 to FR-A4). evidence: an opened coupled clue's mirror tally counts on the
    receiving squad's board (tally's hook). mirrorWindow (bridges 0): every coupled clue's mirror tally does, from the start.
    bridgeRole: the bridges are dedicated (`roleBridges`; the rule is the prompt's, opens and answers stay valid and counted).
    No key: BridgeBoardEnv exactly."""
    def __init__(self, seed, nAgents, evidence=False, bridgeRole=False, mirrorWindow=False, **kw):
        self.evidence, self.bridgeRole, self.mirrorWindow = bool(evidence), bool(bridgeRole), bool(mirrorWindow)
        checkRelay(dict(couple=kw.get('couple', 0.25), bridges=kw.get('bridges', 0), bridgeGuard=kw.get('bridgeGuard', False),
                        evidence=evidence, bridgeRole=bridgeRole, mirrorWindow=mirrorWindow))
        super().__init__(seed, nAgents, **kw)
        self.roleBridges = self.bridgeSet if self.bridgeRole else frozenset()

    def reset(self, seed, nAgents):
        self._rt = {}
        super().reset(seed, nAgents)

    def relayOpened(self, k):
        "Evidence: whether any value has been relayed on k (its clue is then open to the receiving squad)."
        return any(x[2] == k for x in self.relayLog)

    def _shown(self, k):
        return k in self.coupled and (self.mirrorWindow or self.evidence and self.relayOpened(k))

    def relayTally(self, k):
        "[(r, mirror senders, newest)] of clue 1 of k when open to the receiving squad (mirrorTally's order), else []."
        key = (k, len(self.mirrorLog), len(self.relayLog))
        if key not in self._rt:
            if len(self._rt) > 512:
                self._rt = {}
            self._rt[key] = self.mirrorTally(k).get(k, {}).get(1, []) if self._shown(k) else []
        return self._rt[key]

    def relaySenders(self, k, j=1):
        "{r: mirror senders} that tally merges into the receiving squad's count of clue j of k ({} unless j = 1 and shown)."
        return {r: s for r, s, _ in self.relayTally(k)} if j == 1 else {}

    def countedFalse(self):
        "{(k, 1, r): (time, agent)}: each false mirror value a receiving board has shown, with the event that first exposed it."
        out, first = {}, {}
        for x in self.relayLog:
            first.setdefault(x[2], (x[0], x[1]))
        for t, a, k, j, r, _, false in self.mirrorLog:
            if not false or j != 1 or k not in self.coupled or (k, 1, r) in out:
                continue
            if self.mirrorWindow:
                out[(k, 1, r)] = (t, a)
            elif self.evidence and k in first:
                out[(k, 1, r)] = max((t, a), first[k])
        return out

## Tests

The fixtures follow `bridge_board`'s: seed 1800, 64 agents in squads of 8 (liars 0-15, guards 16-31), `WORLD18`. `act` applies a move and posts a share to the blackboard as `TextClueEnv.post` does.

In [ ]:
from explRL.clue_agents import ClueBoard
from explRL.clue_colony import Q, colonyLine, knownColony, tally
from explRL.colony_board import squadView
from explRL.colony_grid import WORLD18

kw, G = dict(WORLD18), range(16, 32)

def mk(seed=1800, **x):
    return RelayBoardEnv(seed, 64, guardSlots=G, squads=8, nLiar=16, couple=0.25, **dict(dict(bridges=1), **x), **kw)

def act(b, ch, i, a, t):
    n = len(b.shareLog)
    out = b.apply(i, a, t)
    if len(b.shareLog) > n:
        x = b.shareLog[-1]
        ch.post(i, x[2], (x[3], b.tasks[x[2]].task.primes[x[3]], x[4]), t)
    return out

def cast(b):
    "The first open coupled subtask k with an honest clue-0 holder, and its people."
    k = next(k for k in b.open if k in b.coupled and max(b.tasks[k].holders[0]) >= 16)
    s, s2 = b.coupled[k]
    task = b.tasks[k]
    h1, h2 = sorted(task.holders[1])
    o = next(x for x in task.holders[0] if x >= 16)
    liars2 = [x for x in range(16) if x % 8 == s2]
    liar = next(x for x in range(16) if x % 8 == s)
    return k, s, s2, task, h1, h2, o, liars2, liar, (b.bridges[s][0] if b.bridges and b.bridges[s] else None)

In [ ]:
# test_relay_off (AC-1, FR-A1): with no relay key a RelayBoardEnv run equals a BridgeBoardEnv run on the same seed: the same
# board, the same logs and rewards, no relay senders and nothing counted false; tally and colonyLine unchanged
for seed in (1800, 1890):
    a = BridgeBoardEnv(seed, 64, guardSlots=G, squads=8, nLiar=16, couple=0.25, bridges=1, **kw)
    b = mk(seed)
    ca, cb = ClueBoard(), ClueBoard()
    k, s, s2, task, h1, h2, o, liars2, liar, br = cast(a)
    (p1, r1), lie = task.task.clue(1), (task.task.clue(1)[1] + 1) % task.task.clue(1)[0]
    moves = [(liars2[0], ('share', (k, 1, lie))), (h1, ('share', (k, 1, r1))), (br, ('share', (k, 1, lie))),
             (o, ('open', k)), (o, ('submit', task.task.x))]
    for t, (i, m) in enumerate(moves, 1):
        assert act(a, ca, i, m, t)[:3] == act(b, cb, i, m, t)[:3]
    for t in range(6, 40):
        a.advance(t), b.advance(t)
    assert repr(a.ledger) == repr(b.ledger) and a.shareLog == b.shareLog and a.mirrorLog == b.mirrorLog and a.relayLog == b.relayLog
    assert a.submits == b.submits and a.tasks.keys() == b.tasks.keys() and b.roleBridges == frozenset()
    assert all(b.relaySenders(k2) == {} for k2 in b.coupled) and b.countedFalse() == {}
    va, vb = squadView(a, o), squadView(b, o)
    assert tally(ca, va) == tally(cb, vb) and colonyLine(va, tally(ca, va)) == colonyLine(vb, tally(cb, vb))

In [ ]:
# test_check_relay (FR-A6, FR-A4): checkRelay's refusals, on the world and on the board; roleBridges; boardClass and relayKw
from explRL.clue_colony import checkOptions
from explRL.bridge_board import BridgeBoardEnv as _B
base = dict(WORLD18, colony=True, guards=16, squads=8)
for w in (dict(mirrorWindow=True, couple=0.25, bridges=1), dict(bridgeRole=True, couple=0.25),
          dict(bridgeRole=True, couple=0.25, bridges=1, bridgeGuard=True), dict(evidence=True, bridges=1),
          dict(mirrorWindow=True)):
    for f in (lambda: checkOptions(dict(base, **w), 64, 16), lambda: checkRelay(w)):
        try:
            f(); raise AssertionError(w)
        except ValueError:
            pass
for w in (dict(evidence=True, couple=0.25, bridges=1), dict(bridgeRole=True, couple=0.25, bridges=1),
          dict(evidence=True, bridgeRole=True, couple=0.25, bridges=1), dict(mirrorWindow=True, couple=0.25, bridges=0)):
    assert checkOptions(dict(base, **w), 64, 16) and boardClass(w) is RelayBoardEnv and relayKw(w)
assert boardClass(dict(couple=0.25, bridges=1)) is _B and boardClass(dict(squads=8)) is None and relayKw(dict(squads=8)) == {}
try:
    mk(mirrorWindow=True); raise AssertionError('window with a bridge')
except ValueError:
    pass
b = mk(bridgeRole=True)
assert b.roleBridges == b.bridgeSet == frozenset(32 + s for s in range(8)) and mk().roleBridges == frozenset()
# the role is the prompt's, not the board's: a dedicated bridge may still open, and the open is logged
k = next(k for k in b.open if b.squadOfTask[k] == 0 and k not in b.coupled)
b.apply(32, ('open', k), 1)
assert b.opened[32] == k and any(r[0] == 32 and r[3] == 'travel' for r in b.ledger)

In [ ]:
# test_evidence_tally (AC-2, AC-3, FR-A2, FR-A5, P23-2): before a relay the clue is closed; after it the whole mirror tally
# counts downstream with mirror senders only (the bridge's own post never counts); a lone relayed value stays unconfirmed;
# a true holder's later mirror post reaches the receiving board in the same turn, contests a relayed lie, and with a second
# true post leads in knownColony; the display tags relayed values with their squad
b, ch = mk(evidence=True), ClueBoard()
k, s, s2, task, h1, h2, o, liars2, liar, br = cast(b)
(p1, r1) = task.task.clue(1)
lie, lie2 = (r1 + 1) % p1, (r1 + 2) % p1
v = squadView(b, o)
act(b, ch, liars2[0], ('share', (k, 1, lie)), 1)                     # a mirror lie
assert b.relaySenders(k) == {} and not b.relayOpened(k) and 1 not in tally(ch, v).get(k, {})
act(b, ch, br, ('share', (k, 1, lie)), 2)                            # the bridge relays it: one mirror sender
assert b.relayOpened(k) and b.relaySenders(k) == {lie: frozenset({liars2[0]})}
tal = tally(ch, v)
assert tal[k][1] == [(lie, frozenset({liars2[0]}), 1.0)]            # the bridge's post is dropped, the liar's counts
line = colonyLine(v, tal)
assert f'x mod {p1} = {lie} (1 agent in squad {s2}, relayed, unconfirmed)' in line and 'contested' not in line.split(f'T{k}:')[1].split(';')[0]
assert knownColony(o, v, ch)[k][1] == lie                          # shown and unconfirmed: the only value
# AC-3: a true holder's mirror post appears at once (same turn), and the clue is contested
act(b, ch, h1, ('share', (k, 1, r1)), 3)
tal = tally(ch, v)
assert {r for r, *_ in tal[k][1]} == {lie, r1} and colonyLine(v, tal).count(f'in squad {s2}, relayed') == 2
assert f'T{k}: ' in colonyLine(v, tal) and ', contested' in colonyLine(v, tal).split(f'T{k}:')[1].split(';')[0]
act(b, ch, h2, ('share', (k, 1, r1)), 4)
tal = tally(ch, v)
assert tal[k][1][0][:2] == (r1, frozenset({h1, h2})) and knownColony(o, v, ch)[k][1] == r1
assert f'x mod {p1} = {r1} (2 agents in squad {s2}, relayed, confirmed)' in colonyLine(v, tal)
# AC-2: a receiving-squad liar's lone value loses to the relayed confirmed truth; it still counts as one sender there
act(b, ch, liar, ('share', (k, 1, lie2)), 5)
tal = tally(ch, v)
assert tal[k][1][0][0] == r1 and (lie2, frozenset({liar}), 5.0) in tal[k][1] and knownColony(o, v, ch)[k][1] == r1
assert f'x mod {p1} = {lie2} (1 agent, unconfirmed)' in colonyLine(v, tal)
# review: a receiving-squad liar repeating a relayed one-sender lie is shown, not hidden behind the mirror count
act(b, ch, next(x for x in range(16) if x % 8 == s and x != liar), ('share', (k, 1, lie)), 6)
assert f'x mod {p1} = {lie} (1 agent in squad {s2} and 1 in your squad, relayed, confirmed)' in colonyLine(v, tally(ch, v))
# a second board: a relay of a value with 2 true mirror senders is confirmed on arrival
b, ch = mk(evidence=True), ClueBoard()
k, s, s2, task, h1, h2, o, liars2, liar, br = cast(b)
v = squadView(b, o)
for t, i in ((1, h1), (1, h2)):
    act(b, ch, i, ('share', (k, 1, r1)), t)
act(b, ch, br, ('share', (k, 1, r1)), 2)
assert tally(ch, v)[k][1] == [(r1, frozenset({h1, h2}), 1.0)] and f'(2 agents in squad {s2}, relayed, confirmed)' in colonyLine(v, tally(ch, v))
# other squads and the uncoupled subtasks are untouched; the mirror squad's own view never shows its mirror as a subtask
assert all(k not in tally(ch, squadView(b, x)) for x in (h1, liars2[0]))

In [ ]:
# test_window (AC-4, FR-A3): on a window board there is no bridge, and every mirror value appears on the receiving board from
# its first post, with its mirror senders; mirror posts never reach the channel
b, ch = mk(mirrorWindow=True, bridges=0), ClueBoard()
assert b.bridges == [[]] * 8 and b.bridgeSet == frozenset() and b.roleBridges == frozenset()
k, s, s2, task, h1, h2, o, liars2, liar, br = cast(b)
(p1, r1), lie = task.task.clue(1), (task.task.clue(1)[1] + 1) % task.task.clue(1)[0]
v = squadView(b, o)
assert b.relaySenders(k) == {} and 1 not in tally(ch, v).get(k, {})
act(b, ch, liars2[0], ('share', (k, 1, lie)), 1)
act(b, ch, h1, ('share', (k, 1, r1)), 1)
assert not ch.posts.get(k) and b.relaySenders(k) == {lie: frozenset({liars2[0]}), r1: frozenset({h1})}
tal = tally(ch, v)
assert {(r, s_) for r, s_, _ in tal[k][1]} == {(lie, frozenset({liars2[0]})), (r1, frozenset({h1}))}
assert colonyLine(v, tal).count(f'1 agent in squad {s2}, relayed, unconfirmed') == 2
act(b, ch, h2, ('share', (k, 1, r1)), 2)
assert knownColony(o, v, ch)[k][1] == r1 and b.relaySenders(k, 0) == {}
assert all(b.relaySenders(k2) == {} for k2 in b.open if k2 not in b.coupled)

In [ ]:
# test_counted_false (AC-10, FR-D1): countedFalse lists each false mirror value a receiving board has shown, with its first
# exposure; falseValues adds it, so contamination counts an honest answer built on a lie that arrived by window or by the
# whole tally after a relay; on C1's boards (no countedFalse) and with no relay key, falseValues is unchanged
from explRL.colony_trace import contaminatedBy, falseValues
b, ch = mk(mirrorWindow=True, bridges=0), ClueBoard()
k, s, s2, task, h1, h2, o, liars2, liar, br = cast(b)
(p1, r1), lie = task.task.clue(1), (task.task.clue(1)[1] + 1) % task.task.clue(1)[0]
act(b, ch, liars2[0], ('share', (k, 1, lie)), 1)
assert b.countedFalse() == {(k, 1, lie): (1.0, liars2[0])} and falseValues(b, True) == {(k, 1, lie): (1.0, liars2[0])}
assert falseValues(b) == {}                                           # the measures of posted lies are unchanged
x = lie                                                               # an answer that agrees with the lie
sub = (2.0, o, k, x, 'wrong')
assert contaminatedBy(b, sub, x, falseValues(b, True)) == [(k, 1, lie)]
# evidence: a lie posted before the first relay is exposed by the relay (the later event); one posted after, by its own post
b, ch = mk(evidence=True), ClueBoard()
k, s, s2, task, h1, h2, o, liars2, liar, br = cast(b)
(p1, r1), lie, lie2 = task.task.clue(1), (task.task.clue(1)[1] + 1) % task.task.clue(1)[0], (task.task.clue(1)[1] + 2) % task.task.clue(1)[0]
act(b, ch, liars2[0], ('share', (k, 1, lie)), 1)
act(b, ch, h1, ('share', (k, 1, r1)), 1)
assert b.countedFalse() == {}                                         # not yet shown downstream
act(b, ch, br, ('share', (k, 1, r1)), 3)
assert b.countedFalse() == {(k, 1, lie): (3.0, br)}
act(b, ch, liars2[1], ('share', (k, 1, lie2)), 4)
assert b.countedFalse() == {(k, 1, lie): (3.0, br), (k, 1, lie2): (4.0, liars2[1])}
assert all(falseValues(b, True)[key] == val for key, val in b.countedFalse().items())
# a relayed lie is in shareLog already (C1's route): falseValues keeps the earlier of the two exposures
b, ch = mk(evidence=True), ClueBoard()
k, s, s2, task, h1, h2, o, liars2, liar, br = cast(b)
act(b, ch, liars2[0], ('share', (k, 1, lie)), 1)
act(b, ch, br, ('share', (k, 1, lie)), 2)
assert falseValues(b, True)[(k, 1, lie)] == (2.0, br) and b.countedFalse()[(k, 1, lie)] == (2.0, br)   # the relay exposed it
a = BridgeBoardEnv(1800, 64, guardSlots=G, squads=8, nLiar=16, couple=0.25, bridges=1, **kw)
act(a, ClueBoard(), liars2[0], ('share', (k, 1, lie)), 1)
assert falseValues(a, True) == {} and not hasattr(a, 'countedFalse')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()